In [ ]:
# Libraries
import cv2
import numpy as np
import time
import os
import random
import pygame
from tensorflow.keras.models import load_model

# Paths
fer_model_path      = r"C:\Users\Bushra Shahid\Desktop\IDVS\FER Model\model\fer_best_model.keras"
mrl_model_path      = r"C:\Users\Bushra Shahid\Desktop\IDVS\MRL Model\model\mrl_best_model.keras"
songs_path          = r"C:\Users\Bushra Shahid\Desktop\IDVS\FER Model\songs"
mrl_alarm_path      = r"C:\Users\Bushra Shahid\Desktop\IDVS\MRL Model\alarm"
distraction_alarm_path = r"C:\Users\Bushra Shahid\Desktop\IDVS\Distraction Model\alarm"

# Load models
fer_model = load_model(fer_model_path)
mrl_model = load_model(mrl_model_path)

# Initialize pygame
pygame.mixer.init()

# Class labels
FER_CLASSES = ['happiness', 'neutral', 'sadness']

print("Models loaded successfully!")
print("Pygame initialized!")

pygame 2.6.1 (SDL 2.28.4, Python 3.11.0)
Hello from the pygame community. https://www.pygame.org/contribute.html
Models loaded successfully!
Pygame initialized!


In [6]:
# Face and eye detectors
face_cascade         = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade          = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye_tree_eyeglasses.xml')

# FER variables
current_emotion      = None
current_song         = None
fer_history          = []
FER_HISTORY_SIZE     = 5

# MRL variables
eyes_closed_start    = None
eyes_open_start      = None
ALARM_THRESHOLD      = 2.0
OPEN_THRESHOLD       = 1.0
mrl_alarm_playing    = False
eye_history          = []
EYE_HISTORY_SIZE     = 5

# Distraction variables
distraction_start    = None
dist_alarm_playing   = False
baseline_face_size   = None
label_history        = []
LABEL_HISTORY_SIZE   = 8
DISTRACTION_THRESHOLD = 2.0

# ---- FER Functions ----
def get_random_song(emotion):
    """Get random song from emotion folder"""
    emotion_folder = os.path.join(songs_path, emotion)
    if not os.path.exists(emotion_folder):
        return None
    songs = [f for f in os.listdir(emotion_folder) if f.endswith('.mp3')]
    if not songs:
        return None
    return os.path.join(emotion_folder, random.choice(songs))

def play_song(emotion):
    """Play song based on emotion"""
    global current_emotion, current_song
    # Play if emotion changed OR no song playing
    if emotion != current_emotion or not pygame.mixer.music.get_busy():
        song_path = get_random_song(emotion)
        if song_path:
            pygame.mixer.music.load(song_path)
            pygame.mixer.music.play(-1)
            current_song    = song_path
            current_emotion = emotion

def predict_fer(face_img):
    """Predict emotion from face"""
    face_resized    = cv2.resize(face_img, (48, 48))
    face_normalized = face_resized / 255.0
    face_input      = face_normalized.reshape(1, 48, 48, 1)
    prediction      = fer_model.predict(face_input, verbose=0)
    happiness_conf  = prediction[0][0]
    neutral_conf    = prediction[0][1]
    sadness_conf    = prediction[0][2]
    if happiness_conf > 0.60:
        return 'happiness', happiness_conf
    if sadness_conf > 0.15:
        return 'sadness', sadness_conf
    return 'neutral', neutral_conf

def get_stable_fer(emotion):
    """Smooth FER prediction"""
    fer_history.append(emotion)
    if len(fer_history) > FER_HISTORY_SIZE:
        fer_history.pop(0)
    return max(set(fer_history), key=fer_history.count)

# ---- MRL Functions ----
def predict_eye_state(eye_img):
    """Predict eye state"""
    eye_resized    = cv2.resize(eye_img, (64, 64))
    eye_normalized = eye_resized / 255.0
    eye_input      = eye_normalized.reshape(1, 64, 64, 1)
    prediction     = mrl_model.predict(eye_input, verbose=0)
    raw            = prediction[0][0]
    if raw > 0.5:
        return 'closed_eyes', raw
    else:
        return 'open_eyes', 1 - raw

def get_stable_eye(eye_state):
    """Smooth eye prediction"""
    eye_history.append(eye_state)
    if len(eye_history) > EYE_HISTORY_SIZE:
        eye_history.pop(0)
    return max(set(eye_history), key=eye_history.count)

def play_mrl_alarm():
    """Play drowsiness alarm"""
    global mrl_alarm_playing
    alarm_files = [f for f in os.listdir(mrl_alarm_path)
                   if f.endswith('.mp3') or f.endswith('.wav')]
    if alarm_files and not mrl_alarm_playing:
        pygame.mixer.music.load(os.path.join(mrl_alarm_path, alarm_files[0]))
        pygame.mixer.music.play(-1)
        mrl_alarm_playing = True

def stop_mrl_alarm():
    """Stop drowsiness alarm"""
    global mrl_alarm_playing
    if mrl_alarm_playing:
        pygame.mixer.music.stop()
        mrl_alarm_playing = False

# ---- Distraction Functions ----
def get_distraction_label(face, frame_shape, baseline):
    """Detect distraction from face position"""
    h, w           = frame_shape[:2]
    fx, fy, fw, fh = face
    face_center_y  = fy + fh // 2
    frame_center_y = h // 2
    offset_y       = face_center_y - frame_center_y
    size_ratio     = fw / baseline if baseline else 1.0
    if size_ratio < 0.65:
        return "LOOKING AWAY"
    elif offset_y < -100:
        return "LOOKING UP"
    elif offset_y > 100:
        return "LOOKING DOWN"
    else:
        return "FORWARD"

def get_stable_label(label):
    """Smooth distraction label"""
    label_history.append(label)
    if len(label_history) > LABEL_HISTORY_SIZE:
        label_history.pop(0)
    return max(set(label_history), key=label_history.count)

def play_dist_alarm():
    """Play distraction alarm"""
    global dist_alarm_playing
    alarm_files = [f for f in os.listdir(distraction_alarm_path)
                   if f.endswith('.mp3') or f.endswith('.wav')]
    if alarm_files and not dist_alarm_playing:
        pygame.mixer.music.load(os.path.join(distraction_alarm_path, alarm_files[0]))
        pygame.mixer.music.play(-1)
        dist_alarm_playing = True

def stop_dist_alarm():
    """Stop distraction alarm"""
    global dist_alarm_playing
    if dist_alarm_playing:
        pygame.mixer.music.stop()
        dist_alarm_playing = False

print("All functions ready!")

All functions ready!


In [7]:
# Start camera
cap = cv2.VideoCapture(0)

# Reset all variables
eyes_closed_start  = None
eyes_open_start    = None
distraction_start  = None
baseline_face_size = None
mrl_alarm_playing  = False
dist_alarm_playing = False

print("Integrated System Started! Press Q to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1,
                                          minNeighbors=5, minSize=(100, 100))

    # ==================== DISTRACTION MODULE ====================
    if len(faces) > 0:
        face           = faces[0]
        fx, fy, fw, fh = face

        # Set baseline
        if baseline_face_size is None or fw > baseline_face_size:
            baseline_face_size = fw

        dist_label = get_distraction_label(face, frame.shape, baseline_face_size)
        dist_label = get_stable_label(dist_label)

        # Reset distraction when face detected
        distraction_start = None
        stop_dist_alarm()

        # Show direction
        dist_color = (0, 255, 0) if dist_label == "FORWARD" else (0, 165, 255)
        cv2.putText(frame, f"Direction: {dist_label}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, dist_color, 2)

    else:
        # Face not detected - start distraction timer
        dist_label = get_stable_label("FACE NOT DETECTED")
        cv2.putText(frame, "FACE NOT DETECTED", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        if distraction_start is None:
            distraction_start = time.time()

        elapsed = time.time() - distraction_start
        if elapsed >= DISTRACTION_THRESHOLD:
            play_dist_alarm()
            cv2.putText(frame, "DISTRACTION ALERT!",
                        (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    # ==================== MRL MODULE ====================
    all_eye_states = []

    if len(faces) > 0:
        face           = faces[0]
        fx, fy, fw, fh = face
        cv2.rectangle(frame, (fx, fy), (fx+fw, fy+fh), (255, 255, 0), 2)

        roi_gray  = gray[fy:fy+int(fh*0.6), fx:fx+fw]
        roi_color = frame[fy:fy+int(fh*0.6), fx:fx+fw]

        eyes = eye_cascade.detectMultiScale(roi_gray, scaleFactor=1.1,
                                            minNeighbors=3, minSize=(20, 20))

        for (ex, ey, ew, eh) in eyes[:2]:
            eye_img    = roi_gray[ey:ey+eh, ex:ex+ew]
            eye_state, confidence = predict_eye_state(eye_img)
            all_eye_states.append(eye_state)
            color = (0, 255, 0) if eye_state == 'open_eyes' else (0, 0, 255)
            cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), color, 2)

    # Eye state logic
    if len(all_eye_states) > 0:
        closed_count = all_eye_states.count('closed_eyes')
        open_count   = all_eye_states.count('open_eyes')
        raw_eye      = 'closed_eyes' if closed_count >= open_count else 'open_eyes'
    else:
        raw_eye = 'closed_eyes'

    stable_eye = get_stable_eye(raw_eye)

    if stable_eye == 'closed_eyes' and len(faces) > 0:
        eyes_open_start = None
        if eyes_closed_start is None:
            eyes_closed_start = time.time()
        elapsed = time.time() - eyes_closed_start
        if elapsed >= 1.0:
            cv2.putText(frame, f"EYES CLOSED: {elapsed:.1f}s",
                        (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        if elapsed >= ALARM_THRESHOLD:
            play_mrl_alarm()
            cv2.putText(frame, "DROWSINESS ALERT!",
                        (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    else:
        eyes_closed_start = None
        if mrl_alarm_playing:
            if eyes_open_start is None:
                eyes_open_start = time.time()
            if time.time() - eyes_open_start >= OPEN_THRESHOLD:
                stop_mrl_alarm()
                eyes_open_start = None
        else:
            eyes_open_start = None
            cv2.putText(frame, "EYES OPEN", (10, 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    # ==================== FER MODULE ====================
    if len(faces) > 0:
        face           = faces[0]
        fx, fy, fw, fh = face
        face_gray      = gray[fy:fy+fh, fx:fx+fw]
        emotion, conf  = predict_fer(face_gray)
        stable_emotion = get_stable_fer(emotion)
        play_song(stable_emotion)

        # Show emotion
        cv2.putText(frame, f"Mood: {stable_emotion} ({conf:.0%})",
                    (10, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 165, 0), 2)

    cv2.imshow('IDVS - Integrated Driver Vision System', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
pygame.mixer.music.stop()
print("System stopped!")

Integrated System Started! Press Q to quit.
System stopped!
